In [1]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 59.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.56.1
    Uninstalling transformers-4.56.1:
      Successfully uninstalled transformers-4.56.1


## Local Inference on GPU
Model page: https://huggingface.co/Qwen/Qwen3-0.6B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/Qwen/Qwen3-0.6B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-0.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Device set to use cuda:0


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': '<think>\nOkay, the user asked, "Who are you?" I need to respond in a friendly and helpful way. Let me start by acknowledging their question. I should mention that I\'m a language model designed to assist them. I should keep the tone positive and open. Maybe add something about being a tool that can help with various tasks. Let me check if there\'s anything else I should include to make the response more engaging. Yeah, that sounds good. Let me put it all together in a concise and welcoming manner.\n</think>\n\nHello! I\'m a language model designed to assist you in conversations. I can help with a wide range of tasks, from answering questions to writing text, translating between languages, or providing support in different contexts. How can I assist you today?'}]}]

In [3]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

<think>
Okay, the user asked, "Who are you?" I need to respond appropriately. Let me start by confirming my identity. I'm an AI assistant designed to help users with their questions.


# Qwen3 Example

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# 准备输入
prompt = "给我简单介绍一下大语言模型。"
messages = [
    {"role": "user", "content": prompt}
]

# 应用聊天模板
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True  # 启用思考模式（默认为True）
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 生成回复
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# 解析思考内容和最终回复
try:
    # 找到 </think> 标记 (token id: 151668)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("思考内容:", thinking_content)
print("最终回复:", content)

`torch_dtype` is deprecated! Use `dtype` instead!


思考内容: <think>
好的，用户让我简单介绍一下大语言模型。首先，我需要明确大语言模型的定义和基本概念。大语言模型，也叫大型语言模型，是人工智能领域的一个重要分支，主要应用于自然语言处理任务。

接下来，我应该从几个关键点来组织回答。首先是模型的基本组成，比如Transformer架构，因为它在处理长文本时表现很好。然后是模型的应用场景，比如翻译、文本生成、问答等。还要提到训练数据的重要性，以及模型的训练过程和优势，比如高效的训练时间和处理能力。

用户可能对大语言模型的运作机制不太清楚，所以需要解释一下。比如，Transformer如何通过自注意力机制来处理长文本，这样模型可以捕捉更复杂的上下文信息。同时，要强调模型的优势，比如在多个任务上的表现，以及实际应用中的价值。

另外，用户可能想知道如何使用这些模型，或者它们的局限性。虽然问题比较简短，但可以稍微提到一些注意事项，比如数据隐私和模型的可解释性问题，这样回答会更全面。

还要注意语言的简洁明了，避免专业术语过多，让普通用户也能理解。最后检查一下有没有遗漏的关键点，确保回答准确且易于理解。
</think>
最终回复: 大语言模型（Large Language Model，LLM）是一种基于深度学习的智能系统，能够理解和生成人类语言。它们的核心原理是通过大量文本数据进行训练，从而学习语言的结构和模式。

### 核心概念：
1. **训练目标**：模型学习语言的语法、词汇、语义等，以便完成任务如翻译、文本生成、问答等。
2. **关键技术**：  
   - **Transformer架构**：通过自注意力机制（Self-Attention）处理长文本，显著提升对复杂上下文的理解能力。  
   - **分布式训练**：利用大规模数据和计算资源（如GPU/TPU）进行训练，加速模型收敛。

### 应用场景：
- **翻译**：将英文翻译成多种语言，或反之。  
- **文本生成**：创作文章、诗歌、产品描述等。  
- **问答系统**：回答用户的问题，提供信息整合。  
- **智能助手**：如Siri、Alexa等，实现语音交互。

### 特点：
- **高效性**：模型能处理长文本，且训练速度快。  
- **泛化能力**：基于大量数据训练，可适应不同语言和场景。  
- **局限性**：对数

# Tools Use

In [5]:
"""
Qwen3-0.6B 实用工具集合
包含多种使用场景的封装函数
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

class Qwen3Assistant:
    """Qwen3 助手类，封装了常用功能"""

    def __init__(self, model_name="Qwen/Qwen3-0.6B", use_gpu=True):
        """
        初始化模型

        Args:
            model_name: 模型名称或本地路径
            use_gpu: 是否使用 GPU（如果可用）
        """
        print("正在加载模型，首次运行需要下载，请稍候...")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 根据是否有 GPU 设置设备
        if use_gpu and torch.cuda.is_available():
            device_map = "auto"
            torch_dtype = "auto"
            print(f"✓ 使用 GPU: {torch.cuda.get_device_name(0)}")
        else:
            device_map = "cpu"
            torch_dtype = "float32"
            print("✓ 使用 CPU")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch_dtype,
            device_map=device_map
        )

        self.history = []
        print("✓ 模型加载完成！\n")

    def chat(self, user_input, enable_thinking=True, show_thinking=True,
             temperature=0.6, max_tokens=2048):
        """
        单轮对话

        Args:
            user_input: 用户输入
            enable_thinking: 是否启用思考模式
            show_thinking: 是否显示思考过程
            temperature: 温度参数（0-1）
            max_tokens: 最大生成长度

        Returns:
            回复内容
        """
        messages = [{"role": "user", "content": user_input}]

        # 应用聊天模板
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        # 生成回复
        with torch.no_grad():  # 推理时不需要梯度
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,  # 温度>0时采样
                top_p=0.95 if enable_thinking else 0.8,
                top_k=20,
            )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        # 解析输出
        thinking_content = ""
        content = ""

        if enable_thinking:
            try:
                index = len(output_ids) - output_ids[::-1].index(151668)
                thinking_content = self.tokenizer.decode(
                    output_ids[:index],
                    skip_special_tokens=True
                ).strip("\n")
                content = self.tokenizer.decode(
                    output_ids[index:],
                    skip_special_tokens=True
                ).strip("\n")
            except ValueError:
                content = self.tokenizer.decode(
                    output_ids,
                    skip_special_tokens=True
                ).strip("\n")
        else:
            content = self.tokenizer.decode(
                output_ids,
                skip_special_tokens=True
            ).strip("\n")

        # 显示思考过程
        if show_thinking and thinking_content:
            print("💭 思考过程:")
            print("-" * 50)
            print(thinking_content)
            print("-" * 50)
            print()

        return content

    def multi_turn_chat(self, enable_thinking=True):
        """
        多轮对话交互模式
        输入 'quit' 或 'exit' 退出
        输入 'clear' 清除历史
        """
        print("开始对话（输入 'quit' 退出，'clear' 清除历史）")
        print("=" * 50)

        while True:
            user_input = input("\n👤 你: ").strip()

            if user_input.lower() in ['quit', 'exit', '退出']:
                print("再见！")
                break

            if user_input.lower() in ['clear', '清除']:
                self.history = []
                print("✓ 历史记录已清除")
                continue

            if not user_input:
                continue

            # 添加用户消息
            self.history.append({"role": "user", "content": user_input})

            # 生成回复
            text = self.tokenizer.apply_chat_template(
                self.history,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=enable_thinking
            )

            model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                generated_ids = self.model.generate(
                    **model_inputs,
                    max_new_tokens=2048,
                    temperature=0.7,
                    do_sample=True,
                    top_p=0.9,
                    top_k=20,
                )

            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
            response = self.tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

            # 添加助手消息（不包含思考内容）
            self.history.append({"role": "assistant", "content": response})

            print(f"\n🤖 助手: {response}")

    def batch_process(self, questions, enable_thinking=False):
        """
        批量处理多个问题

        Args:
            questions: 问题列表
            enable_thinking: 是否启用思考模式

        Returns:
            回复列表
        """
        results = []
        for i, question in enumerate(questions, 1):
            print(f"处理 {i}/{len(questions)}: {question[:50]}...")
            answer = self.chat(question, enable_thinking=enable_thinking, show_thinking=False)
            results.append({"question": question, "answer": answer})
        return results


# ============ 使用示例 ============

if __name__ == "__main__":
    # 创建助手实例
    assistant = Qwen3Assistant()

    # 示例1: 单轮对话（启用思考模式）
    print("【示例1: 数学推理】")
    response = assistant.chat(
        "如果一个正方形的面积是 25 平方米，它的对角线长度是多少？",
        enable_thinking=True,
        show_thinking=True
    )
    print(f"🤖 回答: {response}\n")

    # 示例2: 单轮对话（关闭思考模式，更快速）
    print("\n【示例2: 快速问答】")
    response = assistant.chat(
        "Python 中如何读取文件？",
        enable_thinking=False,
        show_thinking=False
    )
    print(f"🤖 回答: {response}\n")

    # 示例3: 批量处理
    print("\n【示例3: 批量处理】")
    questions = [
        "什么是人工智能？",
        "机器学习的三种类型是什么？",
        "解释一下神经网络"
    ]
    results = assistant.batch_process(questions)
    for i, result in enumerate(results, 1):
        print(f"\nQ{i}: {result['question']}")
        print(f"A{i}: {result['answer'][:100]}...")

    # 示例4: 多轮对话（取消注释以使用）
    # print("\n【示例4: 多轮对话】")
    # assistant.multi_turn_chat(enable_thinking=False)

正在加载模型，首次运行需要下载，请稍候...
✓ 使用 GPU: Tesla T4
✓ 模型加载完成！

【示例1: 数学推理】
💭 思考过程:
--------------------------------------------------
<think>
嗯，我现在要解决的问题是：如果一个正方形的面积是25平方米，它的对角线长度是多少？好，让我仔细想想这个问题应该怎么解决。

首先，我需要回忆一下正方形的面积和对角线之间的关系。正方形的面积公式我记得应该是边长的平方，也就是面积S等于边长a乘以边长a，也就是a²。对吧？所以，如果面积是25平方米，那边长a应该是多少呢？

让我先算一下边长。面积S = a²，所以a = √S。这里S是25平方米，所以a应该是√25，也就是5米。对吧？所以边长是5米。

接下来，我需要求正方形的对角线长度。正方形的对角线长度，我记得有一个公式，对吧？应该是边长乘以√2。也就是说，对角线d = a√2。这样的话，如果边长是5米的话，对角线长度应该是5乘以√2，也就是大约5×1.414，大约是7.07米。不过题目可能需要精确的结果，还是用根号表达？

不过让我再仔细检查一下，确保自己没有搞错。正方形的对角线确实和边长的关系是√2倍的关系，对吗？比如，想象一个正方形，边长是a，那么对角线的长度应该比边长大，而且是√2倍。所以没错的话，这里的结果是对的。

不过，为了确认，我可以画一个直角坐标系，把正方形的顶点放在坐标原点和(5,0)、(5,5)和(0,5)的位置。这样正方形的对角线是从(0,5)到(5,0)，长度可以用距离公式计算，即√[(5-0)² + (0-5)²] = √[25 + 25] = √50 = 5√2，也就是5乘以√2，对的。所以结果是对的。

所以，对角线长度是5√2米。不过题目可能希望以数值形式给出，或者保留根号。不过题目里没有特别说明，可能需要写成精确形式，也就是5乘以根号2。不过有时候题目可能希望用近似值，但这里面积已经给出是精确的25，所以应该用精确值。

不过让我再检查一遍计算过程有没有哪里出错。面积是25，边长是√25=5，没错。对角线长度是5√2，没错。所以答案应该是5√2米，或者约7.07米。不过题目可能需要精确的结果，所以用根号的话更准确。

不过，可能有人会问，为什么不是直接用面积的平方根？比如

# Qwen3 + Perplexity 集成

# 模拟API

In [8]:
"""
最简单的 Qwen3 + 免费搜索实现
三种方法，从最简单到完全零依赖
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# tools = []
# functioncall = []

# ============ 方法1: 使用 duckduckgo-search 库（最推荐）============

class SimpleQwenWithSearch:
    """最简单的实现 - 使用 duckduckgo-search 库"""

    def __init__(self):
        print("🚀 初始化 Qwen3 + 免费搜索\n")

        # 加载模型
        print("加载 Qwen3...")
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
        self.model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-0.6B",
            torch_dtype="auto",
            device_map="auto"
        )

        # 初始化搜索
        try:
            from duckduckgo_search import DDGS
            self.search_engine = DDGS()
            print("✓ DuckDuckGo 搜索已启用\n")
        except ImportError:
            self.search_engine = None
            print("⚠️  未安装 duckduckgo-search，搜索功能不可用")
            print("安装命令: pip install duckduckgo-search\n")

    def search(self, query):
        """搜索功能"""
        if not self.search_engine:
            return "搜索功能未启用，请安装: pip install duckduckgo-search"

        print(f"🔍 搜索: {query}\n")

        try:
            results = list(self.search_engine.text(query, max_results=5))

            # 格式化结果
            formatted = "搜索结果：\n\n"
            for i, r in enumerate(results, 1):
                formatted += f"{i}. {r['title']}\n"
                formatted += f"   {r['body']}\n"
                formatted += f"   来源: {r['href']}\n\n"

            return formatted
        except Exception as e:
            return f"搜索出错: {e}"

    def chat(self, user_input):
        """简单对话"""
        print(f"💬 你: {user_input}\n")

        # 判断是否需要搜索
        search_keywords = ['最新', '现在', '今天', '当前', '2025', '新闻', '价格']
        needs_search = any(kw in user_input for kw in search_keywords)

        if needs_search and self.search_engine:
            # 搜索并回答
            search_result = self.search(user_input)

            prompt = f"""基于以下搜索结果回答问题。

问题：{user_input}

{search_result}

请简洁准确地回答。"""
        else:
            # 直接回答
            prompt = user_input

        # 生成回复
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.7,
                do_sample=True
            )

        response = self.tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):],
            skip_special_tokens=True
        ).strip()

        print(f"🤖 助手: {response}\n")
        return response


# ============ 方法2: 纯 requests 实现（零额外依赖）============

class MinimalQwenWithSearch:
    """最小依赖实现 - 只需要 requests"""

    def __init__(self):
        import requests
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

        # 加载模型
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
        self.model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-0.6B",
            torch_dtype="auto",
            device_map="auto"
        )

    def search(self, query):
        """使用 DuckDuckGo API（无需额外库）"""
        import json
        from urllib.parse import quote_plus

        print(f"🔍 搜索: {query}\n")

        try:
            url = f"https://api.duckduckgo.com/?q={quote_plus(query)}&format=json"
            response = self.session.get(url, timeout=10)
            data = response.json()

            result = ""

            # 主要答案
            if data.get('AbstractText'):
                result += f"摘要: {data['AbstractText']}\n\n"

            # 相关主题
            if data.get('RelatedTopics'):
                result += "相关信息:\n"
                for i, topic in enumerate(data['RelatedTopics'][:5], 1):
                    if isinstance(topic, dict) and 'Text' in topic:
                        result += f"{i}. {topic['Text']}\n"

            return result if result else "未找到相关结果"

        except Exception as e:
            return f"搜索失败: {e}"

    def chat(self, user_input):
        """对话功能"""
        print(f"💬 你: {user_input}\n")

        # 简单判断
        if any(kw in user_input for kw in ['最新', '现在', '今天']):
            search_result = self.search(user_input)
            prompt = f"基于以下信息回答:\n\n{search_result}\n\n问题: {user_input}"
        else:
            prompt = user_input

        # 生成回复
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024)

        response = self.tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):],
            skip_special_tokens=True
        ).strip()

        print(f"🤖 助手: {response}\n")
        return response


# ============ 方法3: 完全模拟（开发测试用）============

class MockQwenWithSearch:
    """模拟实现 - 用于离线开发测试"""

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
        self.model = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen3-0.6B",
            torch_dtype="auto",
            device_map="auto"
        )

        # 模拟搜索数据库 (Vector Database, MySQL)
        self.mock_db = {
            "人工智能": "人工智能（AI）是计算机科学的一个分支，近年来发展迅速。2025年的主要趋势包括大语言模型、多模态AI和AI安全。",
            "机器学习": "机器学习是AI的子集，包括监督学习、无监督学习和强化学习三大类型。",
            "Python": "Python是一种高级编程语言，广泛应用于数据科学、Web开发和AI领域。",
            "深度学习": "深度学习基于神经网络，特别是深度神经网络。主要应用包括图像识别、自然语言处理等。"
        }

    def search(self, query):
        """模拟搜索"""
        print(f"🎭 模拟搜索: {query}\n")

        # 查找匹配关键词
        for keyword, info in self.mock_db.items():
            if keyword in query:
                return f"模拟搜索结果:\n{info}"

        return f"模拟搜索结果:\n关于「{query}」的信息（这是测试数据）"

    def chat(self, user_input):
        """对话功能"""
        print(f"💬 你: {user_input}\n")

        # 总是使用搜索（用于测试）
        search_result = self.search(user_input)
        prompt = f"{search_result}\n\n基于上述信息，回答: {user_input}"

        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024)

        response = self.tokenizer.decode(
            outputs[0][len(inputs.input_ids[0]):],
            skip_special_tokens=True
        ).strip()

        print(f"🤖 助手: {response}\n")
        return response


# ============ 使用示例 ============

if __name__ == "__main__":
    print("=" * 70)
    print("Qwen3 + 免费搜索 - 三种实现方法")
    print("=" * 70 + "\n")

    # 方法选择
    print("请选择实现方法：")
    print("1. 方法1 - 使用 duckduckgo-search 库（推荐，需先安装）")
    print("2. 方法2 - 纯 requests 实现（零额外依赖）")
    print("3. 方法3 - 完全模拟（离线测试）")
    print()

    # 默认使用方法1
    choice = input("输入选择 (1/2/3，默认1): ").strip() or "1"
    print()

    # 初始化对应的助手
    if choice == "1":
        print("🌟 使用方法1: duckduckgo-search 库\n")
        assistant = SimpleQwenWithSearch()
    elif choice == "2":
        print("🌟 使用方法2: 纯 requests 实现\n")
        assistant = MinimalQwenWithSearch()
    else:
        print("🌟 使用方法3: 完全模拟\n")
        assistant = MockQwenWithSearch()

    # 演示示例
    print("=" * 70)
    print("示例1: 无需搜索的问题")
    print("=" * 70 + "\n")
    assistant.chat("什么是机器学习？")

    print("=" * 70)
    print("示例2: 需要搜索的问题")
    print("=" * 70 + "\n")
    assistant.chat("人工智能的最新发展")

    print("=" * 70)
    print("示例3: 交互模式")
    print("=" * 70 + "\n")

    # 交互模式
    print("进入交互模式（输入 'quit' 退出）\n")
    while True:
        user_input = input("你: ").strip()
        if user_input.lower() in ['quit', 'exit', '退出', 'q']:
            print("\n再见！")
            break
        if user_input:
            assistant.chat(user_input)


# ============ 快速开始代码 ============

"""
快速开始（复制以下代码即可使用）：

# 1. 安装依赖（选择一种）
pip install transformers torch duckduckgo-search  # 方法1
pip install transformers torch requests           # 方法2
pip install transformers torch                    # 方法3

# 2. 使用
from simple_qwen_search import SimpleQwenWithSearch

assistant = SimpleQwenWithSearch()
assistant.chat("你的问题")

# 3. 或者直接运行
python simple_qwen_search.py
"""

Qwen3 + 免费搜索 - 三种实现方法

请选择实现方法：
1. 方法1 - 使用 duckduckgo-search 库（推荐，需先安装）
2. 方法2 - 纯 requests 实现（零额外依赖）
3. 方法3 - 完全模拟（离线测试）

输入选择 (1/2/3，默认1): 1

🌟 使用方法1: duckduckgo-search 库

🚀 初始化 Qwen3 + 免费搜索

加载 Qwen3...
⚠️  未安装 duckduckgo-search，搜索功能不可用
安装命令: pip install duckduckgo-search

示例1: 无需搜索的问题

💬 你: 什么是机器学习？

🤖 助手: <think>
嗯，用户问的是什么是机器学习。首先，我需要确保自己对机器学习的基本概念和定义有准确的理解。机器学习是人工智能的一个分支，主要研究如何让计算机从数据中学习并做出预测或决策。那接下来，我应该详细解释这个概念，可能还要举一些例子，比如图像识别、推荐系统之类的。

然后，用户可能想知道机器学习的应用领域，所以需要涵盖一些常见的例子，比如医疗诊断、金融分析、自然语言处理等等。同时，可能需要提到机器学习的几个主要类型，比如监督学习、无监督学习、深度学习，这样用户能更全面地了解。

还要注意用户可能的潜在需求，比如他们可能对机器学习不太熟悉，或者想了解如何开始学习这个领域。这时候需要简明扼要地说明学习机器学习的方法和资源，比如推荐一些在线课程或者书籍，这样用户可以根据自己的需求进行下一步。

另外，用户可能关心机器学习的优缺点，比如它是否适合复杂问题，或者是否存在过拟合等问题。这时候需要简要提到这些挑战，但不要过于深入，保持回答的简洁。

最后，确保回答结构清晰，分点说明，让用户容易理解。可能还需要提醒用户，机器学习是一个不断发展的领域，未来可能会有更多创新应用。这样用户不仅了解了机器学习的基本定义，还知道如何进一步学习和应用。
</think>

机器学习是人工智能的核心分支，旨在让计算机通过分析数据来实现预测或决策的能力。其核心思想是让算法“学会”从经验中学习，而不是手动编程。以下是关键点：

1. **核心目标**  
   通过数据训练，模型自动识别模式并做出预测（如分类、回归或聚类）。

2. **主要类型**  
   - **监督学习*

'\n快速开始（复制以下代码即可使用）：\n\n# 1. 安装依赖（选择一种）\npip install transformers torch duckduckgo-search  # 方法1\npip install transformers torch requests           # 方法2\npip install transformers torch                    # 方法3\n\n# 2. 使用\nfrom simple_qwen_search import SimpleQwenWithSearch\n\nassistant = SimpleQwenWithSearch()\nassistant.chat("你的问题")\n\n# 3. 或者直接运行\npython simple_qwen_search.py\n'